# Experiment 1: Reparametrization symmetry in deep linear networks

This notebook demonstrates the core claim of the article: a deep linear network has a massive $\mathrm{GL}(n)$ reparametrization symmetry. For any invertible matrix $K$, the weights $(W_1, W_2, W_3)$ and $(K W_1, W_2 K^{-1}, W_3)$ compute the exact same function, because $K$ and $K^{-1}$ cancel between adjacent layers. We show this symmetry holds exactly for linear networks and *breaks* when nonlinear activations are introduced.

**Key prediction from the theory:**
- Linear network: $\|f(x) - f_{\text{reparam}}(x)\| \approx 0$ (machine precision)
- ReLU network: $\|f(x) - f_{\text{reparam}}(x)\| \gg 0$ (symmetry broken)


In [ ]:
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

torch.manual_seed(42)
np.random.seed(42)
print("Setup complete.")


## 1. Deep linear network: GL(n) symmetry

A 3-layer linear network computes $f(x) = W_3 W_2 W_1 x$. For any invertible $K$, the reparametrization $(W_1, W_2, W_3) \mapsto (K W_1,\, W_2 K^{-1},\, W_3)$ leaves the function unchanged: $W_3 (W_2 K^{-1})(K W_1) = W_3 W_2 W_1$. The $K$ and $K^{-1}$ cancel between adjacent layers.


In [ ]:
# Network dimensions
d_in, d_hidden, d_out = 5, 8, 3

# Random weights
W1 = torch.randn(d_hidden, d_in, dtype=torch.float64)
W2 = torch.randn(d_hidden, d_hidden, dtype=torch.float64)
W3 = torch.randn(d_out, d_hidden, dtype=torch.float64)

# Random input
x = torch.randn(d_in, dtype=torch.float64)

# Original output (linear: W3 @ W2 @ W1 @ x)
y_original = W3 @ W2 @ W1 @ x

# Test with 10 random invertible K matrices
n_trials = 10
linear_diffs = []
relu_diffs = []

print(f"{'Trial':>6} | {'Linear diff':>14} | {'ReLU diff':>14}")
print("-" * 42)

for i in range(n_trials):
    # Random invertible K (random matrix is almost surely invertible)
    K = torch.randn(d_hidden, d_hidden, dtype=torch.float64)
    K_inv = torch.linalg.inv(K)
    
    # Reparametrized weights: (W1, W2, W3) -> (K W1, W2 K^{-1}, W3)
    W1_new = K @ W1
    W2_new = W2 @ K_inv
    
    # Linear network: should be identical
    y_reparam_linear = W3 @ W2_new @ W1_new @ x
    diff_linear = torch.norm(y_original - y_reparam_linear).item()
    linear_diffs.append(diff_linear)
    
    # ReLU network: symmetry should break
    y_relu_original = W3 @ torch.relu(W2 @ torch.relu(W1 @ x))
    y_relu_reparam = W3 @ torch.relu(W2_new @ torch.relu(W1_new @ x))
    diff_relu = torch.norm(y_relu_original - y_relu_reparam).item()
    relu_diffs.append(diff_relu)
    
    print(f"{i+1:>6} | {diff_linear:>14.2e} | {diff_relu:>14.4f}")

print(f"\n{'Mean':>6} | {np.mean(linear_diffs):>14.2e} | {np.mean(relu_diffs):>14.4f}")


## 2. Results

The linear network differences are at machine precision ($\sim 10^{-15}$), confirming exact $\mathrm{GL}(n)$ symmetry. The ReLU network differences are orders of magnitude larger, confirming that the nonlinearity breaks the symmetry.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_pos = np.arange(n_trials)
width = 0.35

bars1 = ax.bar(x_pos - width/2, linear_diffs, width, label='Linear (GL(n) symmetry)',
               color='#B8D4E3', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x_pos + width/2, relu_diffs, width, label='ReLU (symmetry broken)',
               color='#C47A5A', edgecolor='white', linewidth=0.5)

ax.set_yscale('log')
ax.set_xlabel('Trial (different random K)')
ax.set_ylabel('Output difference ||f(x) - f_reparam(x)||')
ax.set_title('Reparametrization symmetry: linear vs ReLU networks')
ax.set_xticks(x_pos)
ax.set_xticklabels([str(i+1) for i in range(n_trials)])
ax.legend()

# Add annotation
ax.axhline(y=1e-14, color='#B8D4E3', linestyle='--', alpha=0.5)
ax.text(n_trials - 1, 2e-14, 'machine precision', ha='right', fontsize=9,
        color='#B8D4E3', style='italic')

plt.tight_layout()
plt.savefig('Ex1_fig_reparametrization_symmetry.png', dpi=200, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("Figure saved: Ex1_fig_reparametrization_symmetry.png")


## 3. Verifying the algebra directly

We can also verify that the total weight product $W_3 W_2 W_1$ is unchanged by reparametrization, independent of any input.


In [ ]:
# Direct verification: W_total should be identical
W_total_original = W3 @ W2 @ W1

print("Checking that W_total is invariant under reparametrization:\n")
for i in range(5):
    K = torch.randn(d_hidden, d_hidden, dtype=torch.float64)
    K_inv = torch.linalg.inv(K)
    W_total_reparam = W3 @ (W2 @ K_inv) @ (K @ W1)
    diff = torch.norm(W_total_original - W_total_reparam).item()
    print(f"  Trial {i+1}: ||W_total - W_total_reparam|| = {diff:.2e}")

print(f"\nAll differences at machine precision, confirming K^{{-1}}K = I cancellation.")


## Summary

| Network type | Reparametrization symmetry | Typical output difference |
|---|---|---|
| Deep linear | Full $\mathrm{GL}(n)$ | $\sim 10^{-15}$ (machine precision) |
| ReLU | Broken for general $K$ | $\sim 10^{0}$ (large) |

The deep linear network's $\mathrm{GL}(n)$ symmetry means the map from parameters to functions is wildly non-injective: infinitely many weight configurations compute the same linear map. Introducing ReLU shatters this symmetry, leaving only the smaller centralizer $\mathcal{D}^+ \rtimes S_n$ (positive scaling and permutations). This is the first step in the symmetry reduction hierarchy described in the article.
